# Structured output
### Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

# Pydantic
### Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [4]:
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1138c5d10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x113a17d10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year movie was released")
    director:str=Field(description="The director of the movie")
    ratings:float=Field(description="The movies ratings out of 10")
    

In [6]:
model_with_structure = model.with_structured_output(Movie)

In [7]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x1138c5d10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x113a17d10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'ratings': {'description': 'The movies ratings out of 10', 'type':

In [9]:
response = model_with_structure.invoke("Provide details about the movie inception")

In [10]:
response

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

In [14]:
response.ratings

8.8

## Message output alongside parsed structure

In [15]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This year movie was released")
    director:str=Field(..., description="The director of the movie")
    ratings:float=Field(..., description="The movies ratings out of 10")
    

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure.invoke("Provide details about the movie inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception". Let me see what I need to do here. They provided a function called Movie with parameters like director, ratings, title, and year. The required fields are title, year, director, and ratings.\n\nFirst, I need to recall the information about "Inception". The title is obviously "Inception". It was released in 2010. The director is Christopher Nolan. As for ratings, I think it\'s around 8.8 on IMDb, but the function expects a number, so maybe I should use that. Wait, the ratings parameter is a number out of 10, so 8.8 would be appropriate. Let me double-check the release year to make sure. Yes, Inception came out in 2010. Director is definitely Nolan. Ratings might vary, but 8.8 is a commonly cited score. \n\nI need to structure the function call correctly. The required parameters are title, year, director, and ratings. So I\'ll map those values into the J

In [19]:
response['raw'].usage_metadata

{'input_tokens': 223,
 'output_tokens': 333,
 'total_tokens': 556,
 'output_token_details': {'reasoning': 285}}

# Nested Structure

In [30]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name:str
    role:str


class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)

response = model_with_structure.invoke("Provide details about the Ghilli tamil movie ")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the Tamil movie "Ghilli". Let me start by recalling what I know about this movie. Ghilli is a 2004 Tamil action thriller starring Vijay and Shilpa Shetty. The director is S. P. Vijayan. The plot revolves around a man who is framed for a crime he didn\'t commit and has to clear his name.\n\nFirst, I need to check if the MovieDetails function can handle this request. The function requires title, year, cast, genres, and optionally budget. The user didn\'t specify the year, but I know it\'s 2004. The main cast includes Vijay as the lead, Shilpa Shetty, and others like Prakash Raj and Vadivelu. Genres would be action and thriller. Budget might not be readily available, so I can leave it as null.\n\nI should structure the function call with the title "Ghilli", year 2004, cast members with their roles, and the genres. Let me make sure all required fields are included. The required

In [31]:
response['parsed']

MovieDetails(title='Ghilli', year=2004, cast=[Actor(name='Vijay', role='Ghilli'), Actor(name='Shilpa Shetty', role='Meenakshi'), Actor(name='Prakash Raj', role='Antagonist'), Actor(name='Vadivelu', role='Comedy Support')], genres=['Action', 'Thriller'], budget=None)

In [32]:
for x in response['parsed']:
    print(x)

('title', 'Ghilli')
('year', 2004)
('cast', [Actor(name='Vijay', role='Ghilli'), Actor(name='Shilpa Shetty', role='Meenakshi'), Actor(name='Prakash Raj', role='Antagonist'), Actor(name='Vadivelu', role='Comedy Support')])
('genres', ['Action', 'Thriller'])
('budget', None)


# TypedDict

It provides a simpler alternative using python's built-in typing, ideal when you don't need runtime validation

In [33]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict): # no run time validation 
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_with_typedict = model.with_structured_output(MovieDict)

response = model_with_typedict.invoke("Please provide the details of the movie avengers")
response


{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [34]:

class Actor(TypedDict):
    name:str
    role:str


class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails, include_raw=True)

response = model_with_structure.invoke("Provide details about the Ghilli tamil movie ")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user is asking for details about the Tamil movie "Ghilli". First, I need to recall if I have any information on this movie. From what I remember, "Ghilli" is a 2004 Tamil action film starring Vijay, who is a popular actor in Tamil cinema. The movie was directed by S. P. Muthuraj and produced by S. P. Adithya. It\'s possible that the user wants details like the cast, director, budget, and release year.\n\nLooking at the tools provided, there\'s a function called MovieDetails that returns information about a movie. The required parameters are title, year, cast, genres, and budget. I need to check if I can fill these in for "Ghilli". \n\nThe title is clearly "Ghilli". The year is 2004. The cast includes Vijay as the lead actor, and I think he plays the role of a forest officer. Other cast members might include actors like Nadia, who was a popular actress at the time. The director is S. P. Muthuraj